# Baseline YOLO11n sur Pascal VOC

Évaluation **avant fine-tuning** sur le split test du projet, en 640 pixels. Dans Colab, choisir **Exécution > Modifier le type d'exécution > GPU**, puis exécuter les cellules dans l'ordre. Le téléchargement des données fait environ 2,2 Go.

Les résultats Colab sont une nouvelle mesure : pour comparer la vitesse avant/après, refaire les deux évaluations avec le même GPU, les mêmes versions et les mêmes paramètres.

## 1. Installer les dépendances et vérifier le GPU

On conserve la version d'Ultralytics utilisée pour la baseline locale. Colab fournit PyTorch et CUDA ; leurs versions sont affichées et enregistrées plus bas.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.216', 'PyYAML', 'gdown'], check=True)

import torch
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA indisponible : activez un GPU dans les paramètres Colab, puis relancez les cellules.")
print('GPU :', torch.cuda.get_device_name(0))
print('PyTorch :', torch.__version__, '| CUDA :', torch.version.cuda, '| Ultralytics :', ultralytics.__version__)

## 2. Récupérer le projet

Le script et le `data.yaml` partagés sont récupérés depuis `main`. Dans un environnement déjà utilisé, le notebook refuse de modifier un dépôt existant : redémarrer avec une nouvelle session Colab si nécessaire.

In [ ]:
from pathlib import Path
import os

repo = Path('/content/projet_detection_objets_yolo11')
if repo.exists():
    raise FileExistsError(f'{repo} existe déjà ; utilisez une nouvelle session Colab pour éviter de modifier des fichiers existants.')
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/oehtgit/projet_detection_objets_yolo11.git', str(repo)], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit du projet :', commit)

## 3. Télécharger les données préparées

L'archive de la personne 1 est vérifiée par son empreinte SHA-256 avant extraction. Les fichiers restent dans `/content/datasets`, hors du dépôt. Si l'archive Drive n'est pas accessible, demander à la personne 1 de vérifier les droits de partage.

In [ ]:
import hashlib
import zipfile
import gdown
import yaml

dataset_dir = Path('/content/datasets')
archive = dataset_dir / 'VOC_pret_70_10_20.zip'
voc = dataset_dir / 'VOC'
if archive.exists() or voc.exists() or (dataset_dir / 'VOC_coco_ids').exists():
    raise FileExistsError('Des données VOC existent déjà dans /content/datasets ; utilisez une nouvelle session Colab.')
dataset_dir.mkdir(parents=True, exist_ok=True)
downloaded = gdown.download(id='1JD_19qsvIUdjzPuSI2k6zhK-OIr2wMbs', output=str(archive), quiet=False)
if not downloaded:
    raise RuntimeError('Téléchargement impossible : vérifiez les droits du fichier Google Drive.')
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

archive_hash = sha256_file(archive)
expected_archive_hash = 'a049d8f1fa21caa3e95da2634040ca1134bcc99cfac2efd99ff28c345301ae07'
if archive_hash != expected_archive_hash:
    raise ValueError(f'Archive différente de celle utilisée localement : SHA-256 {archive_hash}')
with zipfile.ZipFile(archive) as zf:
    zf.extractall(dataset_dir)

with (repo / 'data/data.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
if Path(config['path']) != voc or config.get('test') != 'images/test':
    raise ValueError('Le data.yaml ne correspond pas au dataset ou au split test attendu.')
images = list((voc / 'images/test').glob('*.jpg'))
labels = list((voc / 'labels/test').glob('*.txt'))
if len(images) != 4300 or len(labels) != 4300:
    raise ValueError(f'Split test incomplet : {len(images)} images et {len(labels)} labels.')
print('Archive vérifiée ; split test :', len(images), 'images et', len(labels), 'labels.')

## 4. Évaluer le modèle préentraîné

Le script du projet réindexe les annotations VOC vers les identifiants COCO pour évaluer correctement les poids préentraînés. Le YAML d'entrée est le même que pour l'entraînement ; l'évaluation utilise le split **test**, 640 pixels et un batch de 16.

Le script télécharge `yolo11n.pt` si nécessaire et crée un dossier `VOC_coco_ids` à côté du dataset. Les fichiers de la session Colab sont temporaires.

In [ ]:
from ultralytics import YOLO, settings

settings.update({'sync': False, 'clearml': False, 'comet': False, 'dvc': False, 'mlflow': False, 'raytune': False})
YOLO('yolo11n.pt')
weights_hash = sha256_file(repo / 'yolo11n.pt')
expected_weights_hash = '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1'
if weights_hash != expected_weights_hash:
    raise ValueError(f'Poids différents de la baseline locale : SHA-256 {weights_hash}')
output_dir = Path('/content/baseline_yolo11n_resultats')
subprocess.run([
    sys.executable, 'data/baseline_zeroshot.py', '--model', 'yolo11n.pt',
    '--data', 'data/data.yaml', '--split', 'test', '--imgsz', '640',
    '--batch', '16', '--device', '0', '--out', str(output_dir),
], check=True)

## 5. Consulter et télécharger les résultats

La vitesse est la moyenne par image pendant la validation en **batch de 16**, et non la latence complète d'une vidéo. Les figures Ultralytics se trouvent dans `runs/val/baseline_zeroshot/` dans la session ; elles disparaissent à sa fermeture.

In [ ]:
import json
import platform
from google.colab import files

metrics_path = output_dir / 'baseline_metrics.json'
with metrics_path.open(encoding='utf-8') as file:
    metrics = json.load(file)
for key in ('precision', 'recall', 'mAP50', 'mAP50-95'):
    print(f'{key} : {metrics[key]:.4f}')
print(f"Inférence : {metrics['speed_ms']['inference']:.2f} ms/image (batch 16)")

metadata = {
    'split': 'test', 'imgsz': 640, 'batch': 16, 'data': 'data/data.yaml',
    'git_commit': commit, 'archive_sha256': archive_hash, 'weights_sha256': weights_hash,
    'python': platform.python_version(), 'ultralytics': ultralytics.__version__,
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
}
metadata_path = output_dir / 'conditions.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
files.download(str(metrics_path))
files.download(str(metadata_path))